# Projekt - fáza č. 2

In [27]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.stats.api as sms
import scipy.stats as stats
from sklearn import preprocessing
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import TargetEncoder
sns.set_theme(style="whitegrid")

## 2.1 A – Rozdelenie dát na trénovaciu a testovaciu množinu

Dáta delíme v pomere 80 % train a 20 % test. Od tohto kroku ďalej pracujeme iba s *train* množinou.

Každá jednotka (id_unit) má v dátach viac meraní (v priemere asi 7). Preto delíme podľa jednotiek, nie podľa riadkov - všetky merania jednej jednotky skončia buď v train, alebo v test. Keby sa tá istá jednotka dostala do oboch množín, model by si ju zapamätal a výsledky na teste by vyšli lepšie, než v skutočnosti sú.

Delenie je zároveň stratifikované na `fail`, takže podiel porúch (asi 12 %) je v oboch množinách rovnaký. `random_state` je pevne daný, aby bolo delenie vždy rovnaké.

In [12]:
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_STATE = 42
TEST_SIZE = 0.20

data = pd.read_csv("./data/readings.csv", sep="\t")
data["id_unit"] = data["id_unit"].str[1:].astype(int)
data.loc[data["id_unit"] > 2000, "id_unit"] -= 90000

sgkf = StratifiedGroupKFold(n_splits=round(1 / TEST_SIZE), shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(sgkf.split(data, data["Failure"], groups=data["id_unit"]))
train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()

assert set(train["id_unit"]).isdisjoint(test["id_unit"])
assert len(train) + len(test) == len(data)

for nazov, cast in [("celé dáta", data), ("train", train), ("test", test)]:
    print(nazov, "- riadky:", len(cast), "- jednotky:", cast["id_unit"].nunique(),
          "- podiel Failure=1:", round(cast["Failure"].mean(), 4))

train.to_csv("./data/Ciste_data/train_raw.csv", index=False, encoding="utf-8-sig")
test.to_csv("./data/Ciste_data/test_raw.csv", index=False, encoding="utf-8-sig")

celé dáta - riadky: 14052 - jednotky: 1999 - podiel Failure=1: 0.1188
train - riadky: 11244 - jednotky: 1597 - podiel Failure=1: 0.1189
test - riadky: 2808 - jednotky: 402 - podiel Failure=1: 0.1186


In [13]:
train.info()

<class 'pandas.DataFrame'>
Index: 11244 entries, 1 to 14051
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_unit             11244 non-null  int64  
 1   cycle_ts            11244 non-null  str    
 2   currentMotor        11244 non-null  float64
 3   dp_filter           10833 non-null  float64
 4   oilTemperature      11244 non-null  float64
 5   pressure_reservoir  11244 non-null  float64
 6   rmsVibration        10820 non-null  float64
 7   Flowmeter           11244 non-null  float64
 8   dischargeDewpoint   11244 non-null  float64
 9   Failure             11244 non-null  int64  
dtypes: float64(7), int64(2), str(1)
memory usage: 1.1 MB


In [14]:
train.head()

,id_unit,cycle_ts,currentMotor,dp_filter,oilTemperature,pressure_reservoir,rmsVibration,Flowmeter,dischargeDewpoint,Failure
1,1922,2023-10-05,5.20071,14.68177,36.67755,8.84326,6.40699,26.62476,7.60072,1
2,548,2023-10-18,5.56161,44.22325,43.50675,8.03135,4.30464,26.39225,5.25673,0
4,298,04 Oct 2023,5.11036,29.84890,64.71004,9.78952,2.82960,33.75524,11.50507,0
5,1522,12 Dec 2023,2.47593,NaN,63.34515,8.66470,5.10853,27.65622,5.79981,1
6,748,2023/04/23,6.60443,18.84023,40.67046,8.61937,NaN,25.53233,3.43388,1


In [15]:
import numpy as np
from sklearn.impute import KNNImputer

def parse_dates(col):
    s = pd.Series(col).astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in ["%Y-%m-%d", "%Y/%m/%d", "%d %b %Y",
                "%m/%d/%Y, %H:%M:%S", "%Y-%m-%d %H:%M:%S"]:
        out = out.fillna(pd.to_datetime(s, format=fmt, errors="coerce"))
    return out

def clean_gateways(df):
    df = df.copy()
    rename_map = {
        "Lon": "lon",
        "idDepot": "id_deport",
        "macAddress": "mac_addr",
        "versionFirmware": "version_firmware",
        "Vendor": "vendor",
        "Technician": "technician",
        "idUnit": "id_unit",
    }
    df = df.rename(columns=rename_map)

    if "gateway_id" not in df.columns:
        df = df.reset_index()
        if "gateway_id" not in df.columns:
            df = df.rename(columns={df.columns[0]: "gateway_id"})

    df["gateway_id"] = pd.to_numeric(
        df["gateway_id"].astype(str).str.replace(r"\D", "", regex=True), errors="coerce"
    ).astype("Int64")

    if "id_unit" in df.columns:
        df["id_unit"] = pd.to_numeric(df["id_unit"], errors="coerce").astype("Int64")

    for c in ["install_date", "seen_last"]:
        df[c] = parse_dates(df[c])
    return df

def clean_readings(df):
    df = df.copy()
    rename_map = {
        "currentMotor": "curr_motor",
        "oilTemperature": "oil_temp",
        "pressure_reservoir": "press_reserv",
        "rmsVibration": "rms_vibration",
        "Flowmeter": "flow",
        "dischargeDewpoint": "dis_dewp",
        "Failure": "fail",
    }
    df = df.rename(columns=rename_map)

    df["id_unit"] = (
        df["id_unit"].astype(str).str.strip().str.replace(r"^U", "", regex=True).str.replace(r"\D", "", regex=True).replace("", np.nan)
    )
    df["id_unit"] = pd.to_numeric(df["id_unit"], errors="coerce")
    bad = df["id_unit"] > 2000
    if bad.any():
        df.loc[bad, "id_unit"] = df.loc[bad, "id_unit"] - 90000
    df["id_unit"] = df["id_unit"].astype("Int64")

    df["cycle_ts"] = parse_dates(df["cycle_ts"])

    for col, bad_value in {"oil_temp": -999, "curr_motor": 99, "dp_filter": 120}.items():
        if col in df.columns:
            df[col] = df[col].replace(bad_value, np.nan)

    sensor_cols = ["curr_motor", "dp_filter", "oil_temp", "press_reserv", "rms_vibration", "flow", "dis_dewp"]
    agg = {c: "mean" for c in sensor_cols}
    agg["fail"] = "max"
    df = df.groupby(["id_unit", "cycle_ts"], as_index=False).agg(agg)
    return df

def clean_units(df):
    df = df.copy()
    rename_map = {
        "idDepot": "id_deport",
        "year_commission": "y_commission",
        "serviceHours": "serv_hours",
        "serial_number": "serial_num",
    }
    df = df.rename(columns=rename_map)

    class_map = {
        "1": 1, "l": 1, "light": 1,
        "2": 2, "med": 2, "n": 2, "normal": 2,
        "3": 3, "h": 3, "heavy": 3,
    }
    if "duty_class" in df.columns:
        raw = df["duty_class"]
        df["duty_class"] = (
            df["duty_class"].astype(str).str.strip().str.lower().map(class_map).astype("Int64")
        )
        assert df["duty_class"].isna().sum() == raw.isna().sum()
    df["overhaul_last"] = parse_dates(df["overhaul_last"])
    df["serv_hours"] = pd.to_numeric(df["serv_hours"], errors="coerce")
    return df

def clean_project_data(gateways, readings, units):
    return clean_gateways(gateways), clean_readings(readings), clean_units(units)

def impute_serv_hours(units):
    u = units.copy()
    u[["y_commission", "serv_hours"]] = KNNImputer(n_neighbors=5).fit_transform(
        u[["y_commission", "serv_hours"]].astype(float))
    return u

WINSOR = {"dp_filter": (0.05, 0.95), "rms_vibration": (0.05, 0.95), "dis_dewp": (0.05, 0.95),
          "press_reserv": (0.05, 0.95), "oil_temp": (0.05, 0.95), "flow": (0.05, 0.95)}

def impute_and_winsorize(readings, limits=WINSOR):
    df = readings.copy()
    for c in ["dp_filter", "rms_vibration"]:
        df[c] = df[c].fillna(df.groupby("fail")[c].transform("median"))
    for c in ["curr_motor", "oil_temp"]:
        df[c] = df[c].fillna(df[c].median())
    for c, (lo, hi) in limits.items():
        df[c] = df[c].clip(df[c].quantile(lo), df[c].quantile(hi))
    return df

def encodeData(data):
    data = data.copy()
    t = pd.to_datetime(data["cycle_ts"], errors="coerce")
    data["cycle_month"] = t.dt.month
    data["cycle_dow"] = t.dt.dayofweek
    data["cycle_hour"] = t.dt.hour
    data["cycle_doy"] = t.dt.dayofyear
    if "y_commission" in data.columns:
        data["unit_age_years"] = t.dt.year - data["y_commission"]
    return data

def build_dataset(gateways, readings, units):
    g, r, u = clean_project_data(gateways, readings, units)
    u = impute_serv_hours(u)
    r = impute_and_winsorize(r)
    df = (r.merge(u.drop(columns="id_deport"), on="id_unit", how="left")
            .merge(g.dropna(subset=["id_unit"]).drop_duplicates("id_unit").drop(columns="id_deport"),
                   on="id_unit", how="left", suffixes=("", "_gw")))
    return encodeData(df)

In [24]:
gateways_c = clean_gateways(pd.read_csv("./data/gateways.csv", sep=";", index_col=13))
units_c    = clean_units(pd.read_csv("./data/units.csv", sep=",", index_col=5))

gateways_c = (gateways_c.dropna(subset=["id_unit"]).drop_duplicates("id_unit")
                        .drop(columns="id_deport"))
gateways_c["id_unit"] = gateways_c["id_unit"].astype("int64")

In [25]:
CAT_COLS  = ["hardware_model", "version_firmware"]
DROP_COLS = ["gateway_id", "mac_addr", "ipv4", "serial_num",
             "install_date", "seen_last", "overhaul_last", "cycle_ts",
             "vendor", "technician"]

def make_numeric_frame(readings_raw):
    r = clean_readings(readings_raw)
    df = (r.merge(units_c, on="id_unit", how="left", validate="m:1")
           .merge(gateways_c, on="id_unit", how="left", validate="m:1"))
    assert len(df) == len(r), "merge zmenil počet riadkov"

    t = df["cycle_ts"]
    df["days_since_install"] = (t - df["install_date"]).dt.days
    df["days_since_overhaul"] = (t - df["overhaul_last"]).dt.days
    df["unit_age_years"] = t.dt.year - df["y_commission"]
    df["cycle_month"] = t.dt.month
    df["cycle_dow"] = t.dt.dayofweek
    df["cycle_doy"] = t.dt.dayofyear
    df["seen_before_install"] = (df["seen_last"] < df["install_date"]).astype(int)
    for c in ["dp_filter", "rms_vibration"]:
        df[c + "_missing"] = df[c].isna().astype(int)

    df = df.drop(columns=DROP_COLS)
    for c in df.columns:
        if str(df[c].dtype) in ("Int64", "Float64"):
            df[c] = df[c].astype("float64")
    return df

train_df = make_numeric_frame(pd.read_csv("./data/Ciste_data/train_raw.csv"))
test_df  = make_numeric_frame(pd.read_csv("./data/Ciste_data/test_raw.csv"))

In [29]:
encoder = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.05,
                           sparse_output=False), CAT_COLS),
     ("depot", TargetEncoder(target_type="binary", smooth="auto", cv=5, random_state=42),
      ["id_deport"])],
    remainder="passthrough", verbose_feature_names_out=False,
).set_output(transform="pandas")

train_num = encoder.fit_transform(train_df, train_df["fail"])
test_num  = encoder.transform(test_df)

In [30]:
assert train_num.select_dtypes(exclude="number").empty
assert list(train_num.columns) == list(test_num.columns)
print(train_num.shape, test_num.shape)
print(train_num.isna().sum()[lambda s: s > 0])

train_num.to_csv("./data/Ciste_data/train_numeric.csv", index=False)
test_num.to_csv("./data/Ciste_data/test_numeric.csv", index=False)

(11016, 33) (2756, 33)
curr_motor         7
dp_filter        406
oil_temp          50
rms_vibration    413
serv_hours       543
dtype: int64


In [34]:
train_num.info()

<class 'pandas.DataFrame'>
RangeIndex: 11016 entries, 0 to 11015
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   hardware_model_GWX-100   11016 non-null  float64
 1   hardware_model_GWX-200   11016 non-null  float64
 2   hardware_model_GWX-300   11016 non-null  float64
 3   hardware_model_NODE-7    11016 non-null  float64
 4   hardware_model_NODE-9    11016 non-null  float64
 5   version_firmware_v1.2.5  11016 non-null  float64
 6   version_firmware_v2.7.0  11016 non-null  float64
 7   version_firmware_v3.4.1  11016 non-null  float64
 8   id_deport                11016 non-null  float64
 9   id_unit                  11016 non-null  float64
 10  curr_motor               11009 non-null  float64
 11  dp_filter                10610 non-null  float64
 12  oil_temp                 10966 non-null  float64
 13  press_reserv             11016 non-null  float64
 14  rms_vibration            10603 no